# Hinglish Sentiment Robustness: XLM-RoBERTa Fine-Tuning & Evaluation

This notebook runs the transformer experimental arm of our study on Google Colab (with a free GPU, e.g. T4).

### Experimental Protocol
- **Model**: `xlm-roberta-base`
- **Methods**: `baseline`, `norm_light`, `aug_A`
- **Seeds**: `1`, `2`, `3` (3 random seeds per method = 9 full runs)
- **Evaluations**: Clean, Family A (0.1, 0.2, 0.3), and Held-Out Family B (0.1, 0.2, 0.3)
- **Artifacts**: Each run produces `robust_xlm-roberta-base_{method}_s{seed}.csv` and saves it immediately to Google Drive.

## Cell 1: Check GPU Environment
Verify that a GPU is allocated and print device specifications.

In [ ]:
!nvidia-smi
import torch
print("PyTorch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("Memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
else:
    print("WARNING: No GPU detected! Go to Runtime -> Change runtime type -> Select T4 GPU.")

## Cell 2: Mount Google Drive
Mount your Google Drive to load datasets and persist experiment CSVs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_DRIVE_DIR = '/content/drive/MyDrive/hinglish_project/results'
os.makedirs(RESULTS_DRIVE_DIR, exist_ok=True)
print("Results will be persisted to:", RESULTS_DRIVE_DIR)

## Cell 3: Clone Repository and Setup Project Tree
Clone the base repository and set up working directory.

In [ ]:
# Clone base repository if not present
if not os.path.exists('/content/hinglish-sentiment'):
    !git clone https://github.com/HxTag/hinglish-sentiment /content/hinglish-sentiment

%cd /content/hinglish-sentiment

# Ensure robust_kit and data directories exist
!mkdir -p robust_kit data/processed results

# Copy data and robust_kit from Drive if uploaded there, or from local upload
# (Adjust the Drive source path if your folder name differs)
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/hinglish_project'
if os.path.exists(f'{DRIVE_PROJECT_PATH}/data/processed'):
    !cp -r {DRIVE_PROJECT_PATH}/data/processed/* data/processed/
    print("Copied processed splits from Drive.")
if os.path.exists(f'{DRIVE_PROJECT_PATH}/robust_kit'):
    !cp -r {DRIVE_PROJECT_PATH}/robust_kit/* robust_kit/
    print("Copied robust_kit from Drive.")
if os.path.exists(f'{DRIVE_PROJECT_PATH}/run_transformer.py'):
    !cp {DRIVE_PROJECT_PATH}/run_transformer.py ./
    print("Copied run_transformer.py from Drive.")

!ls -la data/processed
!ls -la robust_kit

## Cell 4: Install Dependencies
Install required NLP libraries.

In [ ]:
!pip install -q transformers==4.46.3 datasets==3.1.0 accelerate==1.1.1 scikit-learn pandas emoji

## Cell 5: 1-Epoch Smoke Test
Fast verification run with 1 epoch on `baseline` to confirm tokenizer, model loading, training loop, and evaluation without crashing.

In [ ]:
print("=== RUNNING 1-EPOCH SMOKE TEST ===")
!python robust_kit/run_transformer.py --method baseline --seed 1 --epochs 1 --batch_size 16 --out_dir /content/results
print("Smoke test completed successfully!")
!ls -lh /content/results

## Cell 6: Full Experimental Runs (3 Methods x 3 Seeds = 9 Runs)
Executes all 9 full 3-epoch fine-tuning runs in sequence. Immediately copies each result CSV to Google Drive after completion to prevent data loss.

In [ ]:
import shutil

METHODS = ["baseline", "norm_light", "aug_A"]
SEEDS = [1, 2, 3]

for method in METHODS:
    for seed in SEEDS:
        out_filename = f"robust_xlm-roberta-base_{method}_s{seed}.csv"
        drive_dest = os.path.join(RESULTS_DRIVE_DIR, out_filename)
        
        # Check if already completed and saved in Drive
        if os.path.exists(drive_dest):
            print(f"Skipping {method} seed {seed}: already exists at {drive_dest}")
            continue
            
        print(f"\n{'=' * 75}")
        print(f"STARTING METHOD: {method} | SEED: {seed}")
        print(f"{'=' * 75}")
        
        !python robust_kit/run_transformer.py --method {method} --seed {seed} --epochs 3 --batch_size 16 --out_dir /content/results
        
        # Copy to Drive immediately
        local_file = os.path.join("/content/results", out_filename)
        if os.path.exists(local_file):
            shutil.copy(local_file, drive_dest)
            print(f"Successfully saved {out_filename} to Drive: {drive_dest}")
        else:
            print(f"ERROR: {local_file} not found after run!")

print("\nAll 9 experimental runs finished!")

## Cell 7: Verify Output Files on Drive
Confirm that all 9 CSV files are safely stored in Google Drive and display their summary rows.

In [ ]:
import glob
import pandas as pd

saved_files = sorted(glob.glob(os.path.join(RESULTS_DRIVE_DIR, "robust_xlm-roberta-base_*.csv")))
print(f"Found {len(saved_files)}/9 result CSVs in Drive:")
for f in saved_files:
    df = pd.read_csv(f)
    print(f"{os.path.basename(f)}: {len(df)} rows | Clean F1: {df[df['test_condition']=='Clean']['macro_f1'].values[0]:.4f}")